# 2.1 LSTM with target-aware attention

## 1. Business hypothesis and experiment overview

**Owner:** Winston Ho Rui Wei · **Stage 1:** personalised relevance shortlist.

Readers' recent interests change over time. A chronological LSTM encodes each
reader's fixed pre-period history, and target-aware attention selects history
states relevant to each candidate article. Frozen text/image vectors and article
metadata support articles unseen in training; explicit missingness flags and zero
vector placeholders keep eligible articles without vectors in the catalogue.

The model learns click relevance from each request's original recorded slate
(clicked = 1, shown but unclicked = 0). At inference it scores **every article in
the time-eligible union catalogue**, retaining the genuine top 200 for Stage 2
business reranking to five recommendations. Original slates and clicked articles
are never inserted into that shortlist. Unshown outcomes remain unknown.

This notebook has one execution path: bounded Optuna search, apply the winner,
fit on all recorded training requests, restore the validation-selected checkpoint,
export all requests in all three splits, and verify reload from shared Drive.
Selection uses validation macro known-click Recall@100, then observed-click
nDCG@100, then the earlier trial/checkpoint. Final exports additionally report
Recall@200 and observed-click nDCG@200. These are development recovery measures
of exposed clicks, subject to exposure bias and catalogue-coverage limitations.
Project test is the labelled holdout from unused original validation users. Its targets are masked here and never guide model or reranker choices.

Course connection: this adapts the Week 8 sequence-recommendation recipe using embeddings, an LSTM, attention and a dense prediction head. The binary click target uses BCE on logits rather than rating-regression loss.


## 2. Setup and run configuration

In Colab, choose the L4 GPU runtime and upload **`lstm_data.py` and
`lstm_model.py` beside this notebook in `/content`**. Locally, keep both helpers
beside the notebook. Add the shared folder shortcut at the path below.
Run the cells in order. The bounded search is mandatory; final fitting uses every
recorded training request each epoch, and exports cover every train/validation/test
request. The search's 60,000-request epoch cap applies only to tuning.


In [1]:
from pathlib import Path

EXPERIMENT_ID = "2.1_lstm"
OWNER = "Winston Ho Rui Wei"
SEED = 42
FEATURE_CASE = "raw"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
HISTORY_MAX_LEN = 50
SELECTION_K, EXPORT_K, SHORTLIST_SIZE, FINAL_DISPLAY_SIZE = 100, 200, 200, 5
SELECTION_REQUESTS = 100  # fixed BLAKE2b-64 sample, seed42, same rule as 3.1
CATALOGUE_CHUNK = 32768
RELOAD_TOLERANCE = 1e-4
MODEL_HPARAMS = dict(
    article_dim=128, category_dim=16, type_dim=8, topic_dim=16, text_dim=64, image_dim=32,
    lstm_hidden=128, lstm_layers=1, attention_dim=64, interaction_dim=64,
    head_hidden=(256, 128), dropout=0.2, embedding_dropout=0.1,
    use_id_embedding=False, id_dim=16, text_dropout=0.2,
)
TRAIN_HPARAMS = dict(
    max_epochs=8, batch_impressions=256, learning_rate=1e-3, weight_decay=1e-5,
    grad_clip=1.0, early_stopping_patience=2, lr_factor=0.5, lr_patience=0,
    min_lr=1e-5, history_dropout=0.0, use_amp=True, eval_batch_impressions=256,
    max_train_impressions_per_epoch=None,
)
OPTUNA_N_TRIALS, OPTUNA_TIMEOUT_MIN, OPTUNA_EPOCHS = 12, 150, 3
OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH = 60_000
SELECTION_OBJECTIVE = "known_click_recall@100"
SELECTION_TIEBREAK = "observed_click_ndcg@100"


In [2]:
import os, sys, subprocess, platform, json, shutil, time, hashlib, importlib.util, sqlite3
from importlib import metadata
from dataclasses import fields, asdict

if "google.colab" in sys.modules or Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content")
else:
    CODE_DIR = Path.cwd()
for name in ("lstm_data.py", "lstm_model.py"):
    if not (CODE_DIR / name).is_file():
        raise FileNotFoundError(f"Place {name} in {CODE_DIR} before continuing.")

# Keep the runtime's core stack; install only the small missing dependencies.
for dist in ("duckdb", "optuna"):
    try:
        metadata.version(dist)
    except metadata.PackageNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dist], check=True)
sys.path.insert(0, str(CODE_DIR))
for name in ("lstm_data", "lstm_model"):
    sys.modules.pop(name, None)
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import optuna
import lstm_data as D
import lstm_model as M

assert D.EXPORT_SCHEMA_VERSION == 4, "Upload the current schema-v4 helpers."
assert Path(D.__file__).resolve().parent == CODE_DIR.resolve()
assert Path(M.__file__).resolve().parent == CODE_DIR.resolve()
PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT)
SPLIT_MANIFEST = json.loads((DATA_ROOT / "split_manifest.json").read_text())
if SPLIT_MANIFEST.get("protocol") != "ebnerd_small_labelled_user_holdout_v1":
    raise ValueError("Expected the labelled project dataset and its split manifest.")
if json.loads((DATA_ROOT / "feature_engineering" / "split_manifest.json").read_text()) != SPLIT_MANIFEST:
    raise ValueError("Feature outputs and dataset split manifest do not match.")

RUN_ID = pd.Timestamp.now(tz="Asia/Singapore").strftime("%Y%m%d_%H%M%S") + "_seed42"
LOCAL_ROOT = Path("/content/lstm_outputs") if Path("/content").is_dir() else CODE_DIR / "outputs"
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_RUN_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
D.write_json(SPLIT_MANIFEST, RUN_DIR / "split_manifest.json")
PERSIST_RUN_DIR.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
M.set_seed(SEED)
VERSIONS = {dist: metadata.version(dist) for dist in
            ("torch", "numpy", "pandas", "pyarrow", "duckdb", "optuna")}
INPUTS = D.describe_inputs(PATHS, hash_files=False)  # path/size/mtime, not content hashes

def log(*parts):
    print(*parts, flush=True)

def copy_tree(source, destination):
    # Copy completed files atomically. The live Optuna database is outside RUN_DIR.
    for source_file in sorted(Path(source).rglob("*")):
        if source_file.is_file():
            D._copy_atomic(source_file, Path(destination) / source_file.relative_to(source))

log("Run", RUN_ID, "on", DEVICE, "| persistent folder:", PERSIST_RUN_DIR)
log("Dependencies:", VERSIONS)


Mounted at /content/drive
Run 20261104_014914_seed42 on cuda | persistent folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/20261104_014914_seed42
Dependencies: {'torch': '2.11.0+cu130', 'numpy': '2.4.6', 'pandas': '2.2.3', 'pyarrow': '23.0.1', 'duckdb': '1.3.2', 'optuna': '5.0.0'}


## 3. Load and check shared data

Read the existing expanded feature tables without re-exploding them. Preserve
all candidates, exact integer `(user_id, impression_id)` keys, and original labels.
Only request context, article metadata and past history enter prediction; slate
size and later outcomes do not become inference features. Every split uses the
ordered article/vector union **train_val first, then test**, keeping the first
record including missing fields or vectors.


In [3]:
FRAMES, OFFSETS, TABLE_INFO, SPLIT_START = {}, {}, {}, {}
for split in D.SPLITS:
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    offsets = info.pop("offsets")
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())
    if PATHS.get(f"behaviors_{split}"):
        check = D.cross_check_behaviours(frame, PATHS[f"behaviors_{split}"], split)
        assert check["feature_rows_not_in_behaviours"] == 0, check
        assert check["behaviour_candidates_not_in_features"] == 0, check
        assert check.get("label_mismatches", 0) == 0, check
        assert info["impressions"] == check["behaviour_rows"], "Missing original requests"
        info["behaviour_cross_check"] = check
    FRAMES[split], OFFSETS[split], TABLE_INFO[split] = frame, offsets, info
DEVICE_COLUMNS = TABLE_INFO["train"]["device_columns"]
assert all(TABLE_INFO[s]["device_columns"] == DEVICE_COLUMNS for s in D.SPLITS)
assert TABLE_INFO["test"]["labelled"] is False
log("All recorded requests/candidates:",
    {s: (TABLE_INFO[s]["impressions"], len(FRAMES[s])) for s in D.SPLITS})


All recorded requests/candidates: {'train': (232887, 2585747), 'validation': (29110, 344666), 'test': (29142, 348951)}


In [4]:
VOCAB = json.loads(PATHS["feature_vocabulary"].read_text())
CATALOGUE, CATALOGUE_META = D.load_catalogue([PATHS["articles_train_val"], PATHS["articles_test"]])
CATALOGUE_INFO = CATALOGUE_META["info"]   # ordered union: train_val then test, first record wins
SNAPSHOT_IDS = {"train_val": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_train_val"])],
                "test": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_test"])]}
log("Catalogue:", CATALOGUE_INFO)

def expected_dim(kind, default):
    for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
        if PATHS.get(key):
            meta = json.loads(PATHS[key].read_text())
            value = meta.get(f"{kind}_dimension")
            if value and value != default:
                raise ValueError(f"{key}: {kind} dimension {value} != {default}")
    return default
TEXT_DIM, IMAGE_DIM = expected_dim("text", 768), expected_dim("image", 1024)

VECTOR_INFO, VECTOR_SOURCES = {}, {"text": [], "image": []}
for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", TEXT_DIM),
                                ("image", "image_embedding", "has_image_embedding", IMAGE_DIM)):
    for scope in ("train_val", "test"):
        ids, matrix, available, info = D.decode_vectors(PATHS[f"{kind}_{scope}"], column, flag, dim)
        VECTOR_SOURCES[kind].append((ids, matrix, available))
        VECTOR_INFO[f"{kind}_{scope}"] = info
        log(f"{kind:5s} {scope:9s}", {k: v for k, v in info.items() if k != "path"})
TEXT_LOOKUP, text_union = D.union_vectors(VECTOR_SOURCES["text"])
IMAGE_LOOKUP, image_union = D.union_vectors(VECTOR_SOURCES["image"])
for name, union in (("text", text_union), ("image", image_union)):
    VECTOR_INFO[f"{name}_union"] = {k: v for k, v in union.items() if k != "known_ids"}
    log(name, "union:", VECTOR_INFO[f"{name}_union"])
del VECTOR_SOURCES


Catalogue: {'/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/train_val/articles.parquet': {'articles': 20738, 'published_time_missing': 0}, '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/test/articles.parquet': {'articles': 125541, 'published_time_missing': 0}, 'overlapping_ids_first_kept': 20738}
text  train_val {'rows': 20738, 'available': 20720, 'missing': 18, 'dim': 768, 'norm_min_median_max': [0.9999991059303284, 1.0, 1.0000008344650269], 'storage_type': 'list<element: float>', 'reader': 'pyarrow 23.0.1'}
text  test      {'rows': 5619, 'available': 5618, 'missing': 1, 'dim': 768, 'norm_min_median_max': [0.9999991059303284, 1.0, 1.0000008344650269], 'storage_type': 'list<element: float>', 'reader': 'pyarrow 23.0.1'}
image train_val {'rows': 20738, 'available': 16995, 'missing': 3743, 'dim': 1024, 'norm_min_median_max': [0.9999998807907104, 1.0, 1.0000001192092896], 'storage_type': 'fixed_size_list<element: float>[1024]', 'reader': 'duckdb 1.3.2

In [5]:
HISTORIES = {}
for split in D.SPLITS:
    HISTORIES[split] = D.build_history(
        PATHS[f"history_{split}"], SPLIT_START[split], HISTORY_MAX_LEN,
        users_needed=FRAMES[split]["user_id"].to_numpy())
    log(split, "history:", HISTORIES[split].stats)


train history: {'history_users': 15143, 'events_total': 2426247, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 255076, 'max_event_time': '2023-05-18T06:59:59', 'split_start': '2023-05-18T07:00:01', 'max_len': 50, 'users_truncated': 10067, 'users_empty': 0, 'length_quantiles_before_truncation': {0.0: 5.0, 0.5: 93.0, 0.9: 414.0, 1.0: 1896.0}, 'behaviour_users_without_history_row': 0}
validation history: {'history_users': 1719, 'events_total': 252567, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 24847, 'max_event_time': '2023-05-25T06:59:56', 'split_start': '2023-05-25T07:00:15', 'max_len': 50, 'users_truncated': 1121, 'users_empty': 0, 'length_quantiles_before_truncation': {0.0: 5.0, 0.5: 83.0, 0.9: 372.2000000000003, 1.0: 960.0}, 'behaviour_users_without_history_row': 0}
test history: {'history_users': 1757, 'events_total': 262909, 'events_dropped_null_id_or_time': 0, 'read_time_missing': 0, 'scroll_missing': 25269, 'max

## 4. Prepare model inputs

Fit numeric transformations and article statistics on **all training requests and
training histories only**. Reuse that state for validation, test and additional
catalogue pairs. Histories retain the most recent 50 events in chronological
order; padding is masked and histories are fixed before the split starts.
Candidate age is rebuilt from the same article lookup for recorded slates and
catalogue scoring. Missing vectors use zero placeholders plus explicit flags;
missing text/image never removes an otherwise eligible article.


In [6]:
def referenced_ids(split):
    hist = HISTORIES[split]
    mask = np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]
    return np.concatenate([FRAMES[split]["candidate_article_id"].to_numpy(), hist.article_ids[mask]])

all_ids = np.concatenate([referenced_ids(s) for s in D.SPLITS] + [CATALOGUE.index.to_numpy()])
train_ids = referenced_ids("train")
article_stats = D.fit_article_stats(CATALOGUE, train_ids)
ID_VOCAB = {}  # learned article IDs are off; metadata/vectors support unseen articles
ARTICLES, ARTICLE_INFO = D.build_article_table(
    CATALOGUE, all_ids, VOCAB, article_stats, TEXT_LOOKUP, IMAGE_LOOKUP,
    TEXT_DIM, IMAGE_DIM, 8, ID_VOCAB)
PREPROCESSING = {
    "numeric_stats": D.fit_numeric_stats(FRAMES["train"], HISTORIES["train"], OFFSETS["train"], ARTICLES),
    "article_stats": article_stats, "history_max_len": HISTORY_MAX_LEN,
    "device_columns": DEVICE_COLUMNS, "feature_vocabulary": VOCAB,
    "text_dim": TEXT_DIM, "image_dim": IMAGE_DIM, "max_topics": 8,
    "candidate_features": D.CANDIDATE_FEATURE_NAMES,
}
ARRAYS = {s: D.build_split_arrays(s, FRAMES[s], OFFSETS[s], HISTORIES[s], ARTICLES,
                                PREPROCESSING["numeric_stats"], DEVICE_COLUMNS) for s in D.SPLITS}
coverage = {s: D.coverage_report(ARRAYS[s], ARTICLES) for s in D.SPLITS}
for a in ARRAYS.values():
    assert np.isfinite(a.imp_context).all() and np.isfinite(a.cand_features).all()
    assert np.isfinite(a.hist_features).all()
assert ARRAYS["train"].event_names[M.GAP_PREV_FEATURE] == "log_gap_prev_h_z"
assert ARRAYS["train"].event_names[M.GAP_PREV_MISSING] == "gap_prev_missing"
log("Article inputs:", ARTICLE_INFO)
log("Feature order:", {"context": ARRAYS["train"].context_names,
                        "candidate": ARRAYS["train"].candidate_names,
                        "event": ARRAYS["train"].event_names,
                        "article_numeric": ARTICLES.numeric_names})


Article inputs: {'articles': 125541, 'not_in_catalogue': 0, 'category_unknown': 52, 'category_missing': 0, 'type_unknown': 12, 'published_time_missing': 0, 'with_text_vector': 20997, 'with_image_vector': 17243}
Feature order: {'context': ['hour_sin', 'hour_cos', 'dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6', 'is_sso_user', 'is_sso_user_missing', 'is_subscriber', 'is_subscriber_missing', 'is_front_page', 'is_front_page_missing', 'ohe_device_type_000', 'ohe_device_type_001', 'ohe_device_type_002', 'ohe_device_type_missing', 'ohe_device_type_unknown', 'log_history_length_z', 'history_empty'], 'candidate': ['log_age_h_z', 'age_missing'], 'event': ['log_read_time_z', 'read_time_missing', 'scroll_z', 'scroll_missing', 'log_gap_prev_h_z', 'gap_prev_missing', 'log_article_age_at_read_z', 'age_at_read_missing'], 'article_numeric': ['log_body_words_z', 'body_words_missing', 'sentiment_z', 'sentiment_missing', 'premium', 'premium_missing', 'in_catalogue', 'has_text_vector', 'has_i

In [7]:
UNION_IDS = CATALOGUE.index.to_numpy(dtype=np.int64)
POLICY = D.EligibilityPolicy(["train_val/articles.parquet", "test/articles.parquet"])
INDEX = D.build_catalogue_index("all", "union", UNION_IDS, ARTICLES)
ELIGIBILITY = {s: D.eligibility_summary(ARRAYS[s], INDEX) for s in D.SPLITS}
CLICK_ELIGIBILITY = {s: D.clicked_eligibility(ARRAYS[s], INDEX, ARTICLES) for s in D.SPLITS}
VAL = ARRAYS["validation"]
SELECTION_SAMPLE, selection_bids = D.select_requests(
    VAL.user_id, VAL.impression_id, VAL.imp_time_us, "validation", SEED, SELECTION_REQUESTS)
SELECTION = {
    "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
    "earlier_checkpoint_on_equal_metrics": True, "interval": "every epoch",
    "selection_k": SELECTION_K, "pool_policy": POLICY.to_json(),
    "reserved_final_partition": "labelled project test; outcomes deferred until full pipeline freeze",
    "sampling": "smallest BLAKE2b-64(seed:split:behaviour_id), then behaviour_id; chronological output",
    "seed": SEED, "requests": selection_bids, "population_requests": VAL.n_impressions,
    "logged_metrics": [SELECTION_OBJECTIVE, SELECTION_TIEBREAK,
                       "eligible_only_recall@100", "diagnostic_recorded_slate_auc", "ties_at_cutoff"],
}
(RUN_DIR / "selection").mkdir()
D.write_json(SELECTION, RUN_DIR / "selection" / "declaration.json")
pd.DataFrame({"behaviour_id": selection_bids}).to_parquet(
    RUN_DIR / "selection" / "validation_requests.parquet", index=False)
CATALOGUE_FINGERPRINT = hashlib.sha256(UNION_IDS.astype("<i8").tobytes()).hexdigest()
log("Complete pools and scoring work:", ELIGIBILITY)
# Conservative uncompressed estimate for retained score/list rows plus request manifests.
OUTPUT_ESTIMATE_BYTES = 0
for s, a in ARRAYS.items():
    lo, hi = INDEX.bounds(a.imp_time_us)
    retained = int(np.minimum(EXPORT_K, hi - lo).sum())
    estimate = retained * 64 + a.n_candidates * 48 + a.n_impressions * 64
    OUTPUT_ESTIMATE_BYTES += estimate
    log(s, "eligible pairs", ELIGIBILITY[s]["eligible_pairs"], "retained rows", retained,
        "approx uncompressed output GB", round(estimate / 1e9, 3))
log("Output estimate GB:", round(OUTPUT_ESTIMATE_BYTES / 1e9, 3),
    "| local free GB:", round(shutil.disk_usage(RUN_DIR).free / 1e9, 1),
    "| mounted Drive free GB:", round(shutil.disk_usage(PERSIST_RUN_DIR).free / 1e9, 1),
    "(mounted free space may differ from the account quota)")
log("Scoring is exhaustive and may take hours; progress reports runtime/ETA during export.",
    "Catalogue articles without vectors are retained.")
copy_tree(RUN_DIR / "selection", PERSIST_RUN_DIR / "selection")


Complete pools and scoring work: {'train': {'split': 'train', 'snapshot': 'union', 'snapshot_articles': 125541, 'snapshot_excluded_missing_published_time': 0, 'snapshot_with_text_vector': 20997, 'snapshot_with_image_vector': 17243, 'requests': 232887, 'eligible_pairs': 28552935815, 'requests_in_split': 232887, 'empty_pools': 0, 'pool_size_min_median_max': [121981, 122557.0, 123393], 'pair_share_with_text_vector': 0.15265572486981055, 'pair_share_with_image_vector': 0.1275198129394177, 'max_age_hours': None}, 'validation': {'split': 'validation', 'snapshot': 'union', 'snapshot_articles': 125541, 'snapshot_excluded_missing_published_time': 0, 'snapshot_with_text_vector': 20997, 'snapshot_with_image_vector': 17243, 'requests': 29110, 'eligible_pairs': 3608174696, 'requests_in_split': 29110, 'empty_pools': 0, 'pool_size_min_median_max': [123393, 123943.0, 124472], 'pair_share_with_text_vector': 0.15829812276916427, 'pair_share_with_image_vector': 0.13111968539784916, 'max_age_hours': None}

In [8]:
ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
TENSORS = {s: M.SplitTensors(ARRAYS[s], DEVICE) for s in D.SPLITS}
VAL_CLICK_LOOKUP = D.clicks_by_request(CLICK_ELIGIBILITY["validation"])
selection_slates = D.subset_split_arrays(VAL, SELECTION_SAMPLE)
SELECTION_TENSORS = M.SplitTensors(selection_slates, DEVICE)

def model_config(overrides=None):
    sample = ARRAYS["train"]
    gap = PREPROCESSING["numeric_stats"]["log_gap_to_impression_h"]
    params = {**MODEL_HPARAMS, **(overrides or {})}
    return M.ModelConfig(
        variant="attention", n_categories=len(VOCAB["category"]) + 3,
        n_types=len(VOCAB["article_type"]) + 3, n_topics=len(VOCAB["topics"]) + 2,
        n_article_numeric=ARTICLES.numeric.shape[1], text_dim_in=TEXT_DIM, image_dim_in=IMAGE_DIM,
        n_id_buckets=1, n_event_features=sample.hist_features.shape[-1],
        n_candidate_features=sample.cand_features.shape[1], n_context_features=sample.imp_context.shape[1],
        gap_mean=gap["mean"], gap_std=gap["std"], **params)

def select_validation(m):
    scorer = M.CatalogueScorer(m, TENSORS["validation"], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"], chunk=CATALOGUE_CHUNK,
        requests=SELECTION_SAMPLE)
    shortlists, ties = M.catalogue_shortlists(scorer, SELECTION_SAMPLE, SELECTION_K)
    cat = D.known_click_recall(shortlists, CLICK_ELIGIBILITY["validation"], (SELECTION_K,))
    slate = D.impression_metrics(selection_slates.offsets, selection_slates.labels,
        M.predict_logits(m, SELECTION_TENSORS), k=5)
    return {**cat, "ties_at_cutoff": ties, "requests_scored": len(SELECTION_SAMPLE),
            "diagnostic_recorded_slate_auc": slate["mean_impression_auc"]}


## 5. Define and train model

The shared article encoder projects frozen text/image vectors, metadata embeddings
and standardised numeric inputs. A masked chronological LSTM produces history
states; candidate-specific attention (including history recency) builds the user
representation. The head combines user/candidate interactions with request context
and candidate age, returning a click logit. Training uses BCE on observed slate
pairs, AdamW, gradient clipping, a validation-driven learning-rate schedule and
early stopping. Candidate text dropout teaches the metadata/image fallback.

**Mandatory search:** up to 12 trials or a 150-minute timeout checked between
trials (an active trial can finish later), at most three epochs and
60,000 training requests per epoch per trial. Optuna TPE uses seed 42 and median
pruning after three startup trials/one warmup step. Every trial uses the same
model/data seed 42, validation requests and full eligible article pools. The
winner maximises Recall@100, then observed-click nDCG@100, then earlier trial.
Apply all seven winning parameters before final fitting. Final training sees
**all training requests each epoch**, up to eight epochs, patience two. Checkpoint
ties preserve the earlier epoch. Each completed epoch and study snapshot is
copied to the persistent run folder; exact interruption resumption is not claimed.


In [9]:
TUNING_DIR = RUN_DIR / "tuning"
TUNING_DIR.mkdir()
LIVE_DB = LOCAL_ROOT / f"{RUN_ID}_optuna_live.db"  # never copied as a live database
MODEL_SEARCH_KEYS = ("lstm_hidden", "attention_dim", "article_dim", "dropout")
TRAIN_SEARCH_KEYS = ("learning_rate", "weight_decay", "history_dropout")
SEARCH_SPACE = {
    "lstm_hidden": [64, 128, 256], "attention_dim": [32, 64, 128], "article_dim": [64, 128],
    "dropout": [0.1, 0.4], "learning_rate_log": [3e-4, 3e-3],
    "weight_decay_log": [1e-6, 1e-3], "history_dropout": [0.0, 0.3],
}

def study_snapshot(study, trial=None):
    temporary = TUNING_DIR / "snapshot.tmp.db"
    with sqlite3.connect(LIVE_DB) as source, sqlite3.connect(temporary) as destination:
        source.backup(destination)
    os.replace(temporary, TUNING_DIR / "study.db")
    study.trials_dataframe().to_csv(TUNING_DIR / "trials.csv", index=False)
    copy_tree(TUNING_DIR, PERSIST_RUN_DIR / "tuning")

def objective(trial):
    model_overrides = dict(
        lstm_hidden=trial.suggest_categorical("lstm_hidden", SEARCH_SPACE["lstm_hidden"]),
        attention_dim=trial.suggest_categorical("attention_dim", SEARCH_SPACE["attention_dim"]),
        article_dim=trial.suggest_categorical("article_dim", SEARCH_SPACE["article_dim"]),
        dropout=trial.suggest_float("dropout", *SEARCH_SPACE["dropout"]),
    )
    train_overrides = dict(
        learning_rate=trial.suggest_float("learning_rate", *SEARCH_SPACE["learning_rate_log"], log=True),
        weight_decay=trial.suggest_float("weight_decay", *SEARCH_SPACE["weight_decay_log"], log=True),
        history_dropout=trial.suggest_float("history_dropout", *SEARCH_SPACE["history_dropout"]),
    )
    M.set_seed(SEED)  # identical initialisation/data-order seed in every trial
    trial_model = M.build_model(model_config(model_overrides), ARTICLE_TENSORS, DEVICE)
    cfg = M.TrainConfig(**{**TRAIN_HPARAMS, **train_overrides, "max_epochs": OPTUNA_EPOCHS,
                          "max_train_impressions_per_epoch": OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH,
                          "seed": SEED})
    trial_dir = TUNING_DIR / f"trial_{trial.number}"
    try:
        _, best = M.train_model(trial_model, TENSORS["train"], select_validation, SELECTION,
            cfg, trial_dir, resume=False, trial=trial, log=log)
        trial.set_user_attr("best_epoch", best["epoch"])
        trial.set_user_attr(SELECTION_TIEBREAK, best["key"][1])
        return best["key"][0]
    finally:
        # Keep per-epoch history, but discard non-selected trial weights/recovery states.
        if (trial_dir / "training_history.json").is_file():
            shutil.copy2(trial_dir / "training_history.json", TUNING_DIR / f"trial_{trial.number}_history.json")
        shutil.rmtree(trial_dir, ignore_errors=True)
        del trial_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

study = optuna.create_study(
    study_name=f"lstm_recall100_{RUN_ID}", storage=f"sqlite:///{LIVE_DB}", direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1))
study.set_user_attr("selection", SELECTION)
study.set_user_attr("search_space", SEARCH_SPACE)
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT_MIN * 60,
               callbacks=[study_snapshot], gc_after_trial=True)
study_snapshot(study)
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if not completed:
    raise RuntimeError("Optuna completed no trial; no selected configuration is available.")
best_trial = max(completed, key=lambda t: (t.value, t.user_attrs[SELECTION_TIEBREAK], -t.number))
MODEL_HPARAMS.update({k: best_trial.params[k] for k in MODEL_SEARCH_KEYS})
TRAIN_HPARAMS.update({k: best_trial.params[k] for k in TRAIN_SEARCH_KEYS})
TUNING_SUMMARY = {
    "study_name": study.study_name, "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
    "final_tie_rule": "earlier trial", "seed_each_trial": SEED, "search_space": SEARCH_SPACE,
    "sampler": "TPESampler(seed=42)", "pruner": "MedianPruner(startup=3, warmup_steps=1)",
    "budget": {"max_trials": OPTUNA_N_TRIALS, "timeout_min": OPTUNA_TIMEOUT_MIN,
               "max_epochs": OPTUNA_EPOCHS, "train_requests_per_epoch": OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH},
    "actual_trials": len(study.trials), "completed_trials": len(completed),
    "trial_states": {state.name: sum(t.state == state for t in study.trials)
                     for state in {t.state for t in study.trials}},
    "best_trial": best_trial.number, "best_params": best_trial.params,
    "best_recall100": best_trial.value, "best_ndcg100": best_trial.user_attrs[SELECTION_TIEBREAK],
    "study_snapshot": "tuning/study.db", "trials": "tuning/trials.csv",
}
D.write_json(TUNING_SUMMARY, TUNING_DIR / "summary.json")
copy_tree(TUNING_DIR, PERSIST_RUN_DIR / "tuning")
log("Applied winning parameters:", best_trial.params)


[I 2026-11-03 17:50:39,021] A new study created in RDB with name: lstm_recall100_20261104_014914_seed42
/content/lstm_model.py:598: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(batch["labels"])


  epoch 1 step 200/235 loss 0.2985 (8s)
epoch 1: train BCE 0.2965 | val known_click_recall@100 0.4050 observed_click_ndcg@100 0.1219 | diag AUC 0.6554 | lr 1.53e-03 | 12s *best*
  epoch 2 step 200/235 loss 0.2839 (6s)
epoch 2: train BCE 0.2834 | val known_click_recall@100 0.6450 observed_click_ndcg@100 0.1600 | diag AUC 0.6947 | lr 1.53e-03 | 9s *best*
  epoch 3 step 200/235 loss 0.2807 (5s)
epoch 3: train BCE 0.2804 | val known_click_recall@100 0.5300 observed_click_ndcg@100 0.1464 | diag AUC 0.6916 | lr 1.53e-03 | 9s


[I 2026-11-03 17:51:16,481] Trial 0 finished with value: 0.645 and parameters: {'lstm_hidden': 128, 'attention_dim': 32, 'article_dim': 128, 'dropout': 0.2803345035229627, 'learning_rate': 0.0015317709628184257, 'weight_decay': 1.1527987128232402e-06, 'history_dropout': 0.29097295564859826}. Best is trial 0 with value: 0.645.


  epoch 1 step 200/235 loss 0.3151 (6s)
epoch 1: train BCE 0.3110 | val known_click_recall@100 0.3850 observed_click_ndcg@100 0.1237 | diag AUC 0.6691 | lr 4.14e-04 | 9s *best*
  epoch 2 step 200/235 loss 0.2869 (6s)
epoch 2: train BCE 0.2864 | val known_click_recall@100 0.5450 observed_click_ndcg@100 0.1941 | diag AUC 0.6693 | lr 4.14e-04 | 9s *best*
  epoch 3 step 200/235 loss 0.2844 (5s)
epoch 3: train BCE 0.2840 | val known_click_recall@100 0.4950 observed_click_ndcg@100 0.1843 | diag AUC 0.7146 | lr 4.14e-04 | 9s


[I 2026-11-03 17:51:44,540] Trial 1 finished with value: 0.545 and parameters: {'lstm_hidden': 64, 'attention_dim': 128, 'article_dim': 64, 'dropout': 0.2835558684167139, 'learning_rate': 0.00041363293858061296, 'weight_decay': 7.523742884534855e-06, 'history_dropout': 0.1099085529881075}. Best is trial 0 with value: 0.645.


  epoch 1 step 200/235 loss 0.2939 (6s)
epoch 1: train BCE 0.2921 | val known_click_recall@100 0.5150 observed_click_ndcg@100 0.1510 | diag AUC 0.6609 | lr 2.67e-03 | 9s *best*
  epoch 2 step 200/235 loss 0.2803 (6s)
epoch 2: train BCE 0.2799 | val known_click_recall@100 0.5300 observed_click_ndcg@100 0.1213 | diag AUC 0.7444 | lr 2.67e-03 | 9s *best*
  epoch 3 step 200/235 loss 0.2774 (5s)
epoch 3: train BCE 0.2771 | val known_click_recall@100 0.6100 observed_click_ndcg@100 0.1472 | diag AUC 0.7268 | lr 2.67e-03 | 9s *best*


... (16 outputs left out)

[I 2026-11-03 17:54:20,908] Trial 10 finished with value: 0.615 and parameters: {'lstm_hidden': 128, 'attention_dim': 32, 'article_dim': 128, 'dropout': 0.20764968969706762, 'learning_rate': 0.0024562500144140035, 'weight_decay': 5.627964781426105e-05, 'history_dropout': 0.25294031630098024}. Best is trial 0 with value: 0.645.


  epoch 1 step 200/235 loss 0.2981 (6s)
epoch 1: train BCE 0.2960 | val known_click_recall@100 0.4850 observed_click_ndcg@100 0.1378 | diag AUC 0.6469 | lr 1.67e-03 | 9s *best*


[I 2026-11-03 17:54:30,888] Trial 11 pruned. 


Applied winning parameters: {'lstm_hidden': 128, 'attention_dim': 32, 'article_dim': 128, 'dropout': 0.2803345035229627, 'learning_rate': 0.0015317709628184257, 'weight_decay': 1.1527987128232402e-06, 'history_dropout': 0.29097295564859826}


In [10]:
MAIN_CONFIG = model_config()
TRAIN_CONFIG = M.TrainConfig(**{**TRAIN_HPARAMS, "seed": SEED})
assert TRAIN_CONFIG.max_train_impressions_per_epoch is None
M.set_seed(SEED)
model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
TRAINING_DIR = RUN_DIR / "model_training"

def persist_epoch(epoch, row):
    copy_tree(TRAINING_DIR, PERSIST_RUN_DIR / "model_training")

t0 = time.time()
HISTORY, BEST = M.train_model(model, TENSORS["train"], select_validation, SELECTION,
    TRAIN_CONFIG, TRAINING_DIR, resume=False, on_epoch_end=persist_epoch, log=log)
TRAIN_SECONDS = time.time() - t0
model.eval()  # train_model restored best_model_state.pt, not the last epoch
D.write_json(HISTORY, RUN_DIR / "training_history.json")
pd.DataFrame(HISTORY).to_csv(RUN_DIR / "training_history.csv", index=False)
log("Restored selected epoch", BEST["epoch"], "validation selection key", BEST["key"],
    "| final-fit requests per epoch:", ARRAYS["train"].n_impressions)


  epoch 1 step 200/910 loss 0.2986 (5s)
  epoch 1 step 400/910 loss 0.2920 (11s)
  epoch 1 step 600/910 loss 0.2886 (17s)
  epoch 1 step 800/910 loss 0.2864 (22s)
epoch 1: train BCE 0.2853 | val known_click_recall@100 0.5500 observed_click_ndcg@100 0.1432 | diag AUC 0.6957 | lr 1.53e-03 | 28s *best*
  epoch 2 step 200/910 loss 0.2758 (6s)
  epoch 2 step 400/910 loss 0.2758 (11s)
  epoch 2 step 600/910 loss 0.2753 (17s)
  epoch 2 step 800/910 loss 0.2747 (22s)
epoch 2: train BCE 0.2745 | val known_click_recall@100 0.6300 observed_click_ndcg@100 0.1830 | diag AUC 0.7333 | lr 1.53e-03 | 28s *best*
  epoch 3 step 200/910 loss 0.2721 (6s)
  epoch 3 step 400/910 loss 0.2716 (11s)
  epoch 3 step 600/910 loss 0.2709 (17s)
  epoch 3 step 800/910 loss 0.2706 (22s)
epoch 3: train BCE 0.2707 | val known_click_recall@100 0.7300 observed_click_ndcg@100 0.2058 | diag AUC 0.7413 | lr 1.53e-03 | 28s *best*
  epoch 4 step 200/910 loss 0.2690 (6s)
  epoch 4 step 400/910 loss 0.2692 (11s)
  epoch 4 step 6

## 6. Validation results

The checkpoint below was selected on the declared fixed 100-request validation
sample, with every sampled request scored against its full eligible catalogue.
This is repeatedly used development validation. Full validation Recall@100/200,
observed-click nDCG@100/200, original-click eligibility/exclusion coverage and
recorded-slate AUC are calculated during the all-request export in Section 7.
Clicked articles excluded by the pool policy remain in recall denominators and
nDCG ideals. Empty pools return an empty list and zero recovery where clicks exist.
Training/trial histories are saved for the later evaluation notebook's learning
curves, ablations, subgroup diagnostics and explanation work.


In [11]:
log("Optuna outcome:", TUNING_SUMMARY)
log("Selected checkpoint sample metrics:", BEST["metrics"])
log("Selected model:", MAIN_CONFIG.to_json())
display(pd.DataFrame(HISTORY))


Optuna outcome: {'study_name': 'lstm_recall100_20261104_014914_seed42', 'objective': 'known_click_recall@100', 'tiebreak': 'observed_click_ndcg@100', 'final_tie_rule': 'earlier trial', 'seed_each_trial': 42, 'search_space': {'lstm_hidden': [64, 128, 256], 'attention_dim': [32, 64, 128], 'article_dim': [64, 128], 'dropout': [0.1, 0.4], 'learning_rate_log': [0.0003, 0.003], 'weight_decay_log': [1e-06, 0.001], 'history_dropout': [0.0, 0.3]}, 'sampler': 'TPESampler(seed=42)', 'pruner': 'MedianPruner(startup=3, warmup_steps=1)', 'budget': {'max_trials': 12, 'timeout_min': 150, 'max_epochs': 3, 'train_requests_per_epoch': 60000}, 'actual_trials': 12, 'completed_trials': 5, 'trial_states': {'COMPLETE': 5, 'PRUNED': 7}, 'best_trial': 0, 'best_params': {'lstm_hidden': 128, 'attention_dim': 32, 'article_dim': 128, 'dropout': 0.2803345035229627, 'learning_rate': 0.0015317709628184257, 'weight_decay': 1.1527987128232402e-06, 'history_dropout': 0.29097295564859826}, 'best_recall100': 0.645, 'best_n

,epoch,train_bce,known_click_recall@100,observed_click_ndcg@100,eligible_only_recall@100,diagnostic_recorded_slate_auc,ties_at_cutoff,learning_rate,train_seconds,selection_seconds,seconds
0,1,0.285322,0.550,0.143235,0.550,0.695702,0,0.001532,25.193880,2.873580,28.067661
1,2,0.274474,0.630,0.182962,0.630,0.733341,0,0.001532,25.172700,2.867530,28.040430
2,3,0.270688,0.730,0.205845,0.730,0.741258,0,0.001532,25.149393,2.856978,28.006584
3,4,0.268225,0.735,0.197304,0.735,0.716250,0,0.001532,25.241694,2.877481,28.119384
4,5,0.266408,0.715,0.194384,0.715,0.715081,0,0.001532,25.145612,2.897892,28.043713
5,6,0.263525,0.665,0.170285,0.665,0.721438,0,0.000766,25.277560,2.862699,28.140470


## 7. Predict and export all splits

First save the selected model, fitted transforms, feature/category names, article
mappings, frozen article inputs and source helpers, then copy them to shared Drive.
For **all requests**, score the complete eligible union catalogue in article
batches and keep a running top 200. Rank by descending logit then ascending
original article ID; never insert slate candidates or clicked articles. Export
compressed Parquet pair scores and native integer-list rankings, with explicit
empty pools and request eligibility manifests. Save every original slate candidate
separately under `diagnostics/recorded_slates`; labels stay in the original input
and are joined separately. No exhaustive catalogue score file is written.

All split exports use schema v4, selection K=100, export/shortlist K=200 and final
Stage 2 display size=5. Scores are finite click logits; no calibration is claimed.
Training predictions are in-sample diagnostics, unsuitable as out-of-fold stacking
features. Holdout labels stay on disk for final evaluation after pipeline freeze.


In [12]:
MODEL_DIR = RUN_DIR / "model"
(MODEL_DIR / "code").mkdir(parents=True)
M.atomic_torch_save(model.state_dict(), MODEL_DIR / "model_state.pt")
D.write_json(MAIN_CONFIG.to_json(), MODEL_DIR / "model_config.json")
D.write_json(PREPROCESSING, MODEL_DIR / "preprocessing.json")
feature_dictionary = {
    "model_family": "LSTM with target-aware attention",
    "ordered_features": {"request_context": ARRAYS["train"].context_names,
                         "candidate": ARRAYS["train"].candidate_names,
                         "history_event": ARRAYS["train"].event_names,
                         "article_numeric": ARTICLES.numeric_names},
    "category_index_to_value": {**{"0": "padding", "1": "missing", "2": "unknown"},
        **{str(i + 3): v for i, v in enumerate(VOCAB["category"])}},
    "article_type_index_to_value": {**{"0": "padding", "1": "missing", "2": "unknown"},
        **{str(i + 3): v for i, v in enumerate(VOCAB["article_type"])}},
    "topic_index_to_value": {**{"0": "padding", "1": "unknown"},
        **{str(i + 2): v for i, v in enumerate(VOCAB["topics"])}},
    "fitted_transformations": "preprocessing.json; reuse fitted state without refitting",
    "article_row_to_original_id": "mappings.npz: article_table_ids",
    "frozen_article_inputs": "article_inputs.npz; compact frozen vectors, metadata, missingness flags",
    "history_order": "oldest first in retained fixed history; masked padding",
    "missing_vectors": "zero vector placeholder plus has_text_vector/has_image_vector flags",
    "score": "click logit, higher is better; sigmoid would be uncalibrated",
}
D.write_json(feature_dictionary, MODEL_DIR / "feature_dictionary.json")
D.write_json({"policy": POLICY.to_json(), "selection_k": SELECTION_K, "export_k": EXPORT_K,
              "shortlist_size": SHORTLIST_SIZE, "final_display_size": FINAL_DISPLAY_SIZE},
             MODEL_DIR / "catalogue_policy.json")
np.savez_compressed(MODEL_DIR / "mappings.npz", article_table_ids=ARTICLES.article_ids,
                    train_referenced_ids=np.unique(train_ids),
                    id_vocab_ids=np.array([], dtype=np.int64), id_vocab_rows=np.array([], dtype=np.int64))
# Include the frozen lookup so inference does not depend on rebuilding vector snapshots.
np.savez_compressed(MODEL_DIR / "article_inputs.npz",
                    **{f.name: np.asarray(getattr(ARTICLES, f.name)) for f in fields(D.ArticleTable)})
for name in ("lstm_model.py", "lstm_data.py"):
    shutil.copy2(CODE_DIR / name, MODEL_DIR / "code" / name)
(RUN_DIR / "requirements.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n")
copy_tree(MODEL_DIR, PERSIST_RUN_DIR / "model")
for name in ("training_history.json", "training_history.csv", "requirements.txt"):
    D._copy_atomic(RUN_DIR / name, PERSIST_RUN_DIR / name)
log("Selected inference bundle saved to", PERSIST_RUN_DIR / "model")


Selected inference bundle saved to /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/20261104_014914_seed42/model


In [13]:
# Every original recorded candidate, including shown non-clicks and ineligible items.
DIAGNOSTICS, VALIDATION_SLATE_METRICS = {}, None
DIAG_SCHEMA = pa.schema([("behaviour_id", pa.string()), ("article_id", pa.int64()),
                         ("score", pa.float32())])
for split in D.SPLITS:
    a = ARRAYS[split]
    scores = M.predict_logits(model, TENSORS[split], batch_size=TRAIN_CONFIG.eval_batch_impressions)
    assert np.isfinite(scores).all()
    bids = np.repeat(D.behaviour_ids(a.user_id, a.impression_id), np.diff(a.offsets))
    path = RUN_DIR / "diagnostics" / "recorded_slates" / f"{split}_scores.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)
    with pq.ParquetWriter(path, DIAG_SCHEMA, compression="zstd") as writer:
        for start in range(0, len(scores), 100_000):
            stop = min(start + 100_000, len(scores))
            writer.write_table(pa.Table.from_pydict({"behaviour_id": bids[start:stop],
                "article_id": a.cand_article_id[start:stop], "score": scores[start:stop].astype(np.float32)},
                schema=DIAG_SCHEMA))
    DIAGNOSTICS[split] = {"path": str(path.relative_to(RUN_DIR)), "rows": len(scores),
                          "requests": a.n_impressions, "labels_joined_separately": True}
    if split == "validation":
        VALIDATION_SLATE_METRICS = D.impression_metrics(a.offsets, a.labels, scores, k=5)
    D._copy_atomic(path, PERSIST_RUN_DIR / path.relative_to(RUN_DIR))
    log(split, "recorded-slate diagnostic rows", len(scores))
    del bids, scores


train recorded-slate diagnostic rows 2585747
validation recorded-slate diagnostic rows 344666
test recorded-slate diagnostic rows 348951


In [14]:
CATALOGUE_RECORDS = {}
for split in D.SPLITS:
    a = ARRAYS[split]
    bids = D.behaviour_ids(a.user_id, a.impression_id)
    scorer = M.CatalogueScorer(model, TENSORS[split], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"], chunk=CATALOGUE_CHUNK)
    lookup = D.clicks_by_request(CLICK_ELIGIBILITY[split])
    hits, ties = {}, {100: 0, 200: 0}
    writer = D.ShortlistWriter(RUN_DIR, split, EXPORT_K)
    t0 = time.time()
    for i in range(a.n_impressions):
        top_ids, top_scores = scorer.topk(i, EXPORT_K)
        top_ids, top_scores = top_ids.cpu().numpy(), top_scores.cpu().numpy()
        # One genuine-full-pool reference check before discarding other scores.
        if i == 0:
            ids, logits = scorer.score_request(i)
            assert np.array_equal(top_ids, D.top_k_reference(logits.cpu().numpy(), ids.cpu().numpy(), EXPORT_K))
            del ids, logits
        writer.add(bids[i], int(a.imp_time_us[i]), int(scorer.hi[i] - scorer.lo[i]), top_ids, top_scores)
        for k in ties:
            ties[k] += int(scorer.last_ties[k])
        h = D.click_hits(lookup, i, top_ids, (SELECTION_K, EXPORT_K))
        if h is not None:
            hits[i] = h
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            log(split, i + 1, "/", a.n_impressions, "requests | elapsed min", round(elapsed / 60, 1),
                "| ETA min", round(elapsed / (i + 1) * (a.n_impressions - i - 1) / 60, 1))
    files = writer.close()
    check = D.validate_catalogue_exports(a, INDEX, files, EXPORT_K)
    metrics = D.recall_from_hits(hits, CLICK_ELIGIBILITY[split], (SELECTION_K, EXPORT_K),
                                 requests_scored=a.n_impressions)
    CATALOGUE_RECORDS[split] = {
        "scope": "all recorded requests", "files": {k: str(Path(files[k]).relative_to(RUN_DIR))
            for k in ("rankings", "scores", "manifest")}, "coverage": ELIGIBILITY[split],
        "metrics": metrics, "ties_crossing_cutoff": {str(k): v for k, v in ties.items()},
        "seconds": time.time() - t0, "integrity_check": check,
    }
    D.write_json(CATALOGUE_RECORDS[split], RUN_DIR / "catalogue" / f"{split}.json")
    for name in ("rankings", "scores", "manifest"):
        source = Path(files[name])
        D._copy_atomic(source, PERSIST_RUN_DIR / source.relative_to(RUN_DIR))
    D._copy_atomic(RUN_DIR / "catalogue" / f"{split}.json",
                   PERSIST_RUN_DIR / "catalogue" / f"{split}.json")
    log(split, "complete:", check, "retrieval:", metrics)
    del scorer, lookup, hits
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


train 1000 / 232887 requests | elapsed min 0.2 | ETA min 53.0
train 2000 / 232887 requests | elapsed min 0.5 | ETA min 52.9
train 3000 / 232887 requests | elapsed min 0.7 | ETA min 52.7
train 4000 / 232887 requests | elapsed min 0.9 | ETA min 52.0
train 5000 / 232887 requests | elapsed min 1.1 | ETA min 51.7
train 6000 / 232887 requests | elapsed min 1.4 | ETA min 51.6
train 7000 / 232887 requests | elapsed min 1.6 | ETA min 51.4
train 8000 / 232887 requests | elapsed min 1.8 | ETA min 51.2
train 9000 / 232887 requests | elapsed min 2.1 | ETA min 51.0
train 10000 / 232887 requests | elapsed min 2.3 | ETA min 50.8
train 11000 / 232887 requests | elapsed min 2.5 | ETA min 50.6
train 12000 / 232887 requests | elapsed min 2.7 | ETA min 50.4
train 13000 / 232887 requests | elapsed min 3.0 | ETA min 50.2
train 14000 / 232887 requests | elapsed min 3.2 | ETA min 50.0
train 15000 / 232887 requests | elapsed min 3.4 | ETA min 49.8
train 16000 / 232887 requests | elapsed min 3.7 | ETA min 49.6
t

In [15]:
METRICS = {
    "partition": "validation development; repeatedly used for tuning/checkpoint selection",
    "selection": {**SELECTION, "selected_epoch": BEST["epoch"], "selected_key": BEST["key"],
                  "selected_sample_metrics": BEST["metrics"]},
    "full_validation_catalogue": CATALOGUE_RECORDS["validation"]["metrics"],
    "full_validation_recorded_slates": VALIDATION_SLATE_METRICS,
    "definitions": {
        "known_click_recall@K": "macro hits / all original known clicks, including ineligible missed clicks",
        "observed_click_ndcg@K": "macro binary observed-click DCG / ideal over all original known clicks, capped K",
        "unknown_outcomes": "zero observed gain is retrieval bookkeeping, never a negative click label",
        "recorded_slate_auc": "diagnostic mean impression AUC on observed shown-click labels only",
        "ties": "descending score, ascending original article_id; crossing-cutoff ties reported",
    },
    "test": {"source_labelled": True, "evaluation": "deferred until pipeline freeze", "metrics": None},
}
# Uploaded code may have no Git checkout; source helpers themselves are bundled.
CODE_PROVENANCE = {"source": "uploaded/beside-notebook helpers", "commit": None,
                   "uncommitted_changes": None}
try:
    CODE_PROVENANCE["commit"] = subprocess.check_output(
        ["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    CODE_PROVENANCE["uncommitted_changes"] = bool(subprocess.check_output(
        ["git", "-C", str(CODE_DIR), "status", "--porcelain"], stderr=subprocess.DEVNULL, text=True).strip())
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
RUN_CONFIG = {
    "experiment_id": EXPERIMENT_ID, "owner": OWNER, "run_id": RUN_ID, "seed": SEED,
    "python": platform.python_version(), "packages": VERSIONS, "code": CODE_PROVENANCE,
    "input_files": INPUTS, "dataset_split": SPLIT_MANIFEST, "feature_case": FEATURE_CASE, "feature_dictionary": "model/feature_dictionary.json",
    "preprocessing": "model/preprocessing.json", "model": MAIN_CONFIG.to_json(),
    "train_config": asdict(TRAIN_CONFIG), "all_training_requests": ARRAYS["train"].n_impressions,
    "actual_final_epochs": len(HISTORY), "train_seconds": TRAIN_SECONDS,
    "chosen_checkpoint": {"path": "model/model_state.pt", "epoch": BEST["epoch"]},
    "selection": SELECTION, "tuning": TUNING_SUMMARY, "export_schema_version": D.EXPORT_SCHEMA_VERSION,
    "prediction_pool": "time_eligible_catalogue", "pool_policy": POLICY.to_json(),
    "catalogue_id_fingerprint_sha256": CATALOGUE_FINGERPRINT,
    "catalogue_sources": CATALOGUE_INFO, "vector_sources": VECTOR_INFO,
    "vector_duplicate_rule": "train_val then test, first record wins including missing vectors",
    "selection_k": SELECTION_K, "export_k": EXPORT_K, "shortlist_size": SHORTLIST_SIZE,
    "final_display_size": FINAL_DISPLAY_SIZE, "score_semantics": "uncalibrated click logit, higher is better",
    "export_request_scope": "all requests in train/validation/test", "catalogue_exports": CATALOGUE_RECORDS,
    "recorded_slate_exports": DIAGNOSTICS, "feature_coverage": coverage,
    "train_predictions": "in-sample; not out-of-fold", "test_labels": "reserved on disk for final evaluation; masked in development",
    "recovery": "epoch checkpoint under model_training/recovery; exact resumption not checked",
    "persistent_artifact_location": str(PERSIST_RUN_DIR),
}
D.write_json(METRICS, RUN_DIR / "metrics.json")
D.write_json(RUN_CONFIG, RUN_DIR / "run_config.json")
D.write_json({"export_schema_version": D.EXPORT_SCHEMA_VERSION,
              "selection_k": SELECTION_K, "export_k": EXPORT_K,
              "shortlist_size": SHORTLIST_SIZE, "final_display_size": FINAL_DISPLAY_SIZE,
              "prediction_pool": "time_eligible_catalogue", "pool_policy": POLICY.to_json(),
              "splits": CATALOGUE_RECORDS, "recorded_slates": DIAGNOSTICS}, RUN_DIR / "export_manifest.json")
copy_tree(RUN_DIR, PERSIST_RUN_DIR)
# Verify saved-file coverage and sizes once; numerical verification follows.
FILE_COVERAGE = {str(f.relative_to(RUN_DIR)): f.stat().st_size
                 for f in RUN_DIR.rglob("*") if f.is_file()}
assert all((PERSIST_RUN_DIR / name).is_file() and (PERSIST_RUN_DIR / name).stat().st_size == size
           for name, size in FILE_COVERAGE.items())
D.write_json(FILE_COVERAGE, RUN_DIR / "manifest.json")
D._copy_atomic(RUN_DIR / "manifest.json", PERSIST_RUN_DIR / "manifest.json")
log("Full validation development metrics:", METRICS["full_validation_catalogue"])
log("Persistent file coverage/sizes checked:", len(FILE_COVERAGE), "files")


Full validation development metrics: {'requests_with_known_clicks_evaluated': 29110, 'requests_without_known_clicks_scored': 0, 'known_clicks_evaluated': 29249, 'known_clicks_eligible': 29249, 'split_click_eligibility': {'clicked_articles': 29249, 'eligible': 29249, 'eligibility_coverage': 1.0, 'not_in_catalogue_snapshot': 0, 'published_time_missing': 0, 'published_after_request': 0}, 'known_click_recall@100': 0.7273147830069849, 'observed_click_ndcg@100': 0.19670032214437094, 'eligible_only_recall@100': 0.7273147830069849, 'eligible_only_requests@100': 29110, 'known_click_recall@200': 0.8525254780716821, 'observed_click_ndcg@200': 0.21436864847703904, 'eligible_only_recall@200': 0.8525254780716821, 'eligible_only_requests@200': 29110}
Persistent file coverage/sizes checked: 51 files


## 8. Reload check

Load source helpers, the selected weights, frozen article inputs, feature mappings
and fitted preprocessing **from the persistent Drive copy**. Rebuild a small
validation batch using that saved state and the shared fixed history/context,
without refitting or retraining. Compare its logits with the original model at
absolute tolerance 1e-4. The copied helper modules, `model_config.json`,
`preprocessing.json`, `article_inputs.npz` and `mappings.npz` provide reconstruction;
new requests still need the documented pre-request context/history inputs.


In [16]:
bundle = PERSIST_RUN_DIR / "model"
def load_bundled_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

BM = load_bundled_module("saved_lstm_model", bundle / "code" / "lstm_model.py")
BD = load_bundled_module("saved_lstm_data", bundle / "code" / "lstm_data.py")
prep = json.loads((bundle / "preprocessing.json").read_text())
cfg = BM.ModelConfig.from_json(json.loads((bundle / "model_config.json").read_text()))
with np.load(bundle / "article_inputs.npz", allow_pickle=False) as saved:
    article_fields = {f.name: saved[f.name] for f in fields(BD.ArticleTable)}
article_fields["numeric_names"] = article_fields["numeric_names"].tolist()
reload_articles = BD.ArticleTable(**article_fields)
with np.load(bundle / "mappings.npz", allow_pickle=False) as mappings:
    assert np.array_equal(mappings["article_table_ids"], reload_articles.article_ids)
assert json.loads((bundle / "feature_dictionary.json").read_text()) == feature_dictionary
reloaded = BM.build_model(cfg, BM.article_table_tensors(reload_articles, DEVICE), DEVICE)
reloaded.load_state_dict(torch.load(bundle / "model_state.pt", map_location=DEVICE, weights_only=True))
reloaded.eval()
# Reconstruct only a small complete-slate batch using the persisted fitted transforms.
picks = np.sort(SELECTION_SAMPLE[:8])
probe_frame, probe_offsets = BD.subset_requests_frame(FRAMES["validation"], OFFSETS["validation"], picks)
probe_arrays = BD.build_split_arrays("validation", probe_frame, probe_offsets, HISTORIES["validation"],
    reload_articles, prep["numeric_stats"], prep["device_columns"])
original_arrays = D.subset_split_arrays(ARRAYS["validation"], picks)
expected_logits = M.predict_logits(model, M.SplitTensors(original_arrays, DEVICE))
loaded_logits = BM.predict_logits(reloaded, BM.SplitTensors(probe_arrays, DEVICE))
difference = np.abs(expected_logits - loaded_logits)
RELOAD_CHECK = {"source": str(bundle), "requests": len(picks), "candidate_rows": len(difference),
    "max_abs_logit_difference": float(difference.max()), "tolerance": RELOAD_TOLERANCE,
    "fitted_state_reused_without_refitting": True, "persistent_file_count": len(FILE_COVERAGE),
    "passed": bool(len(difference) and np.isfinite(difference).all() and difference.max() <= RELOAD_TOLERANCE)}
D.write_json(RELOAD_CHECK, RUN_DIR / "reload_check.json")
D._copy_atomic(RUN_DIR / "reload_check.json", PERSIST_RUN_DIR / "reload_check.json")
assert RELOAD_CHECK["passed"], RELOAD_CHECK
log("Numerical reload check:", RELOAD_CHECK)


Numerical reload check: {'source': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/20261104_014914_seed42/model', 'requests': 8, 'candidate_rows': 68, 'max_abs_logit_difference': 0.0, 'tolerance': 0.0001, 'fitted_state_reused_without_refitting': True, 'persistent_file_count': 51, 'passed': True}


## 9. Business interpretation and handoff

Interpret the **executed** full-validation Recall@100/200 and observed-click
nDCG@100/200 alongside recorded-slate AUC and eligibility/vector coverage.
Original-click recovery supports a relevance-shortlisting decision; it does not
establish unseen-article CTR, retention or profit. The attention architecture's
incremental benefit needs the planned later ablations and explanation analysis.
Attention weights alone are not causal explanations.

The reranking/evaluation owner receives all three splits' genuine top-200 score
Parquets, matching native-list rankings, all original recorded-slate diagnostic
scores and the complete reloadable model/run bundle. Join original labels using
`behaviour_id = "user_id:impression_id"` and `article_id`. Preserve the Stage 1
score/rank when applying validation-chosen business reranking to five articles.
Project test is labelled, but outcome metrics are deferred until the whole pipeline is frozen. Later cross-model
comparison, subgroup plots, learning curves and ablations belong in the planned
evaluation notebook described in `Experiments/README.md`.


In [17]:
log("Handoff folder:", PERSIST_RUN_DIR)
log("Actual completed tuning trials:", TUNING_SUMMARY["completed_trials"],
    "| final epochs:", len(HISTORY), "| selected epoch:", BEST["epoch"])
log("All-request export counts:", {s: CATALOGUE_RECORDS[s]["integrity_check"] for s in D.SPLITS})
log("Full validation:", METRICS["full_validation_catalogue"])
log("Persistent numerical reload passed:", RELOAD_CHECK["passed"])


Handoff folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/20261104_014914_seed42
Actual completed tuning trials: 5 | final epochs: 6 | selected epoch: 4
All-request export counts: {'train': {'requests': 232887, 'eligible_pairs': 28552935815, 'score_rows': 46577400, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}, 'validation': {'requests': 29110, 'eligible_pairs': 3608174696, 'score_rows': 5822000, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}, 'test': {'requests': 29142, 'eligible_pairs': 3611848189, 'score_rows': 5828400, 'empty_pools': 0, 'all_request_ids_and_pool_counts_checked': True, 'score_ranking_requests_checked': 128}}
Full validation: {'requests_with_known_clicks_evaluated': 29110, 'requests_without_known_clicks_scored': 0, 'known_clicks_evaluated': 29249, 'known_clicks_eligible': 29249, 'split_click_eligibility': 